In [ ]:
import os
from pyspark.sql import functions as F
from src.utils.config import *
from src.ingestion.aviation_edge import fetch_flight_history
from src.ingestion.loaders import load_json_to_spark_df
from src.utils.kafka import create_kafka_consumer, consume_kafka_messages
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)
boostrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')

consumer_instance = create_kafka_consumer(
    client_id='flight_processor',
    bootstrap_servers=boostrap_servers,
    topic_name='raw_aviation_data',
)

In [ ]:
# print(json.dumps(all_flight_data[0], indent=2))

# Consume form Kafka topic and store in a list
all_flights_data = consume_kafka_messages(consumer_instance)

# 4. Parallelize the JSON string into a flight collection RDD
flight_collection = spark.sparkContext.parallelize(all_flight_data, 10)

# 5. Create a DataFrame for analysis

aviation_df = (spark.read.json(flight_collection)
   .select(
        "flight.iataNumber",
        "airline.name",
        "arrival.scheduledTime",
        "arrival.actualTime",
        "arrival.delay",
        "arrival.baggage",
        "departure.scheduledTime",
        "departure.actualTime",
        "departure.delay",
        "codeshared.airline.name",
        "codeshared.flight.iataNumber",)
    .toDF(
        "IATA_number",
        "Airline",
        "Arr_scheduled_time",
        "Arr_actual_time",
        "Arr_delay (mins)",
        "Arr_baggage (mins)",
        "Dep_scheduled_time",
        "Dep_actual_time",
        "Dep_delay (mins)",
        "Codeshared_airline",
        "Codeshared_flight_number")
)

# df.printSchema()
aviation_df.show(n=3, truncate=False)

In [ ]:
# for flight in data:
#     rows_to_insert.append([
#         flight.get('flight', {}).get('iataNumber'),
#         flight.get('airline', {}).get('name'),
#         flight.get('arrival', {}).get('scheduledTime'),
#         flight.get('arrival', {}).get('actualTime'),
#         flight.get('arrival', {}).get('delay'),
#         flight.get('arrival', {}).get('baggage'),
#         flight.get('departure', {}).get('scheduledTime'),
#         flight.get('departure', {}).get('actualTime'),
#         flight.get('departure', {}).get('delay'),
#         flight.get('codeshared', {}).get('airline', {}).get('name'),
#         flight.get('codeshared', {}).get('flight', {}).get('iataNumber')
#     ])

## Insert batch into ClickHouse
# client.insert('aviation_flights', rows_to_insert, column_names=[
#     'IATA_number', 'Airline', 'Arr_scheduled_time', 'Arr_actual_time', 'Arr_delay', 'Arr_baggage',
#     'Dep_scheduled_time', 'Dep_actual_time', 'Dep_delay', 'Codeshared_airline', 'Codeshared_flight_number'
# ])